In [87]:
# Python standard library
import os
import warnings

# Numerical & data processing
import numpy as np
import pandas as pd

# PyTorch
import torch

# Hugging Face / Transformers
from transformers import AutoTokenizer,AutoModelForSequenceClassification,AutoModelForCausalLM,Trainer,TrainingArguments,EarlyStoppingCallback

# PEFT / LoRA
from peft import LoraConfig, get_peft_model

# Datasets
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn.metrics import accuracy_score,f1_score,classification_report
from tqdm import tqdm

# Hugging Face Hub (Kaggle)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Warnings
warnings.filterwarnings("ignore")


In [89]:
# !pip install -U protobuf==3.20.3


**MODULE 1 — Chargement et préparation des données**

Cette étape permet de vérifier la taille des jeux de données et leur structure, ainsi que l’absence de valeurs manquantes ou d’anomalies évidentes.

In [90]:
# Chemins vers les fichiers d'entraînement et de test
test_path = "/kaggle/input/data-test/nli_fr_test.tsv"
train_path = "/kaggle/input/data-train/nli_fr_train.tsv"

# Chargement des données au format TSV
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

# Affichage des dimensions des jeux de données
print("Train set : ", train_df.shape)
print("Test set : ", test_df.shape)

# Aperçu des premières lignes du jeu d'entraînement
train_df.head(5)

# train_df.info()           # structure des données
# train_df.isna().sum()     # valeurs manquantes

# la distribution des classes (optionnelle)
# train_df['label'].value_counts(normalize=True)


Train set :  (5010, 3)
Test set :  (2490, 3)


,-e premise,hypo,label
0,"Eh bien, je ne pensais même pas à cela, mais j...",Je ne lui ai pas parlé de nouveau,contradiction
1,"Eh bien, je ne pensais même pas à cela, mais j...",J'étais si contrarié que je commençais juste à...,entailment
2,"Eh bien, je ne pensais même pas à cela, mais j...",Nous avons eu une grande discussion.,neutral
3,"Et je pensais que c'était un privilège, et ça ...",Je n'avais pas conscience que je n'étais pas l...,neutral
4,"Et je pensais que c'était un privilège, et ça ...",J'avais l'impression que j'étais le seul à avo...,entailment


La tâche est formulée comme un problème de classification à trois classes.
Les étiquettes textuelles sont converties en identifiants numériques afin d’être compatibles avec les modèles Transformers.

In [91]:
# Encodage des relations textuelles en identifiants numériques
label2id = {
    "contradiction": 0,
    "entailment": 1,
    "neutral": 2
}

# Mapping inverse
id2label = {v: k for k, v in label2id.items()}

# Création de la colonne label_id
train_df["label_id"] = train_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

train_df.head()


,-e premise,hypo,label,label_id
0,"Eh bien, je ne pensais même pas à cela, mais j...",Je ne lui ai pas parlé de nouveau,contradiction,0
1,"Eh bien, je ne pensais même pas à cela, mais j...",J'étais si contrarié que je commençais juste à...,entailment,1
2,"Eh bien, je ne pensais même pas à cela, mais j...",Nous avons eu une grande discussion.,neutral,2
3,"Et je pensais que c'était un privilège, et ça ...",Je n'avais pas conscience que je n'étais pas l...,neutral,2
4,"Et je pensais que c'était un privilège, et ça ...",J'avais l'impression que j'étais le seul à avo...,entailment,1


Une validation stratifiée est utilisée afin d’évaluer la capacité de généralisation des modèles tout en conservant une distribution équilibrée des classes.

In [92]:
# Séparation du jeu d'entraînement en train / validation (15 %)
# La stratification garantit une distribution similaire des classes
train_df, valid_df = train_test_split(train_df,test_size = 0.15,stratify = train_df["label_id"],random_state = 42)

print("Train_set : " , train_df.shape)
print("Valid_set : ", valid_df.shape)

# Vérification de la distribution des classes
train_df["label"].value_counts(normalize=True)
valid_df["label"].value_counts(normalize=True)


Train_set :  (4258, 4)
Valid_set :  (752, 4)


label
neutral          0.333777
entailment       0.333777
contradiction    0.332447
Name: proportion, dtype: float64

Les données sont converties au format Dataset de HuggingFace afin de faciliter l’intégration avec les tokenizers et les classes Trainer.
La colonne cible est renommée en labels conformément aux conventions de l’API Transformers.

In [93]:
# Conversion des DataFrames pandas vers des objets Dataset
hf_train = Dataset.from_pandas(train_df[["-e premise", "hypo", "label_id"]].reset_index(drop=True))
hf_valid = Dataset.from_pandas(valid_df[["-e premise", "hypo", "label_id"]].reset_index(drop=True))
hf_test = Dataset.from_pandas(test_df[["-e premise", "hypo", "label_id"]].reset_index(drop=True))

# Renommage de la colonne cible
hf_train = hf_train.rename_column("label_id", "labels")
hf_valid = hf_valid.rename_column("label_id", "labels")
hf_test  = hf_test.rename_column("label_id", "labels")

hf_train

Dataset({
    features: ['-e premise', 'hypo', 'labels'],
    num_rows: 4258
})

**MODULE 2 — Encoder CamemBERT + LoRA**

Le tokenizer CamemBERT est utilisé afin de segmenter les phrases françaises en unités compatibles avec le modèle pré-entraîné.

In [94]:
# Noms des modèles encodeurs utilisés dans le projet
model_name_camembert = "almanach/camembert-base"
model_name_camemberta = "almanach/camemberta-base"

# Chargement du tokenizer associé à CamemBERT
tokenizer = AutoTokenizer.from_pretrained(model_name_camembert)


Les deux phrases sont tokenisées conjointement afin de modéliser explicitement leur relation sémantique.
La longueur maximale est fixée à 256 tokens, offrant un compromis entre performance et coût mémoire.

Après tokenisation, les colonnes textuelles sont supprimées et les données sont converties au format PyTorch pour l’entraînement avec le Trainer.

In [95]:
def tokenize_function(batch):
    return tokenizer(
        batch["-e premise"],
        batch["hypo"],
        padding ="max_length",
        truncation = True,
        max_length = 256
    )

hf_train_tok = hf_train.map(tokenize_function,batched = True)
hf_valid_tok = hf_valid.map(tokenize_function, batched=True)
hf_test_tok  = hf_test.map(tokenize_function, batched=True)

# Suppression des colonnes textuelles devenues inutiles
hf_train_tok = hf_train_tok.remove_columns(["-e premise", "hypo"])
hf_valid_tok = hf_valid_tok.remove_columns(["-e premise", "hypo"])
hf_test_tok  = hf_test_tok.remove_columns(["-e premise", "hypo"])

# Conversion au format PyTorch
hf_train_tok.set_format("torch")
hf_valid_tok.set_format("torch")
hf_test_tok.set_format("torch")

Map:   0%|          | 0/4258 [00:00<?, ? examples/s]

Map:   0%|          | 0/752 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Le modèle CamemBERT est utilisé comme encodeur, auquel est ajoutée une tête de classification pour prédire l’une des trois relations possibles.

L’ajustement du modèle est réalisé à l’aide de la méthode LoRA, qui permet de ne mettre à jour qu’une fraction réduite des paramètres du modèle.
Les adaptateurs sont injectés dans les projections d’attention, réduisant significativement le coût computationnel.

In [96]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name_camembert,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

lora_config_camembert = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config_camembert)
model.print_trainable_parameters()


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at almanach/camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,811 || all params: 111,512,070 || trainable%: 0.7962


Les performances sont évaluées à l’aide de l’accuracy et du score F1 macro, ce dernier permettant de tenir compte d’un éventuel déséquilibre entre les classes.

In [97]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

Le modèle est entraîné avec un taux d’apprentissage de 1e-4 et une régularisation par weight decay. Un entraînement en précision mixte (fp16) est utilisé afin d’accélérer le calcul sur GPU.

Un arrêt anticipé est utilisé afin d’éviter le surapprentissage, les performances se stabilisant après quelques époques.

In [98]:
training_args = TrainingArguments(
    output_dir="./camembert_lora",
    eval_strategy="epoch", 
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_valid_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.099500,1.089382,0.345745,0.198561
2,0.899600,0.880583,0.611702,0.600711
3,0.779200,0.805506,0.668883,0.670005
4,0.699900,0.786755,0.690160,0.690941
5,0.705900,0.767171,0.695479,0.695762
6,0.632700,0.769121,0.695479,0.696000
7,0.647600,0.770402,0.706117,0.705030
8,0.618900,0.767629,0.704787,0.703983
9,0.620600,0.765911,0.707447,0.707427
10,0.578800,0.769829,0.715426,0.715375


TrainOutput(global_step=3204, training_loss=0.7151390275407522, metrics={'train_runtime': 940.4944, 'train_samples_per_second': 67.911, 'train_steps_per_second': 4.258, 'total_flos': 6791700071669760.0, 'train_loss': 0.7151390275407522, 'epoch': 12.0})

**MODULE 2 — CamemBERTa + LoRA & évaluation finale des encodeurs**

Le modèle CamemBERTa est chargé avec une tête de classification afin de prédire l’une des trois relations sémantiques.
Un tokenizer spécifique est utilisé afin de garantir la cohérence avec le pré-entraînement du modèle.

Contrairement à CamemBERT, CamemBERTa repose sur une architecture de type DeBERTa, dont les projections d’attention sont implémentées via les modules query_proj et value_proj.
Les adaptateurs LoRA sont injectés dans ces couches afin d’assurer un ajustement efficace du modèle.

In [99]:
tokenizer_a = AutoTokenizer.from_pretrained(model_name_camemberta)
model_a = AutoModelForSequenceClassification.from_pretrained(
    model_name_camemberta,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

lora_config_camemberta = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model_a = get_peft_model(model_a, lora_config_camemberta)
model_a.print_trainable_parameters()

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at almanach/camemberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 297,219 || all params: 112,694,790 || trainable%: 0.2637


Les paramètres d’entraînement sont maintenus identiques à ceux de CamemBERT afin d’assurer une comparaison équitable entre les deux modèles encodeurs.

Un arrêt anticipé est utilisé afin d’éviter le surapprentissage, les performances se stabilisant rapidement au cours de l’entraînement.

In [100]:
training_args_a = TrainingArguments(
    output_dir="./camemberta_lora",
    eval_strategy="epoch", 
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer_a = Trainer(
    model=model_a,
    args=training_args_a,
    train_dataset=hf_train_tok,
    eval_dataset=hf_valid_tok,
    processing_class=tokenizer_a,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_a.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.100500,1.098319,0.353723,0.268489
2,1.097000,1.094301,0.368351,0.356216
3,1.086600,1.089115,0.369681,0.312028
4,1.065800,1.081101,0.390957,0.339963
5,1.046900,1.075717,0.404255,0.364393
6,1.048000,1.067226,0.416223,0.363674
7,1.066700,1.069505,0.421543,0.378751
8,1.018600,1.074481,0.432181,0.386857
9,1.015900,1.073950,0.426862,0.385123
10,1.033500,1.075877,0.418883,0.376685


TrainOutput(global_step=2670, training_loss=1.06044447395239, metrics={'train_runtime': 1120.3384, 'train_samples_per_second': 38.006, 'train_steps_per_second': 2.383, 'total_flos': 5737103353958400.0, 'train_loss': 1.06044447395239, 'epoch': 10.0})

En complément de l’évaluation intégrée au Trainer, une évaluation manuelle est réalisée sur l’ensemble de test afin de garantir une mesure fiable de la généralisation des modèles.

In [101]:
def eval_encoder(model, tokenizer, dataset, batch_size=32, max_length=256):
    model.eval()
    y_true, y_pred = [], []

    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[i:i+batch_size]

        enc = tokenizer(
            batch["-e premise"],
            batch["hypo"],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()

        y_pred.extend(preds.tolist())
        y_true.extend(batch["labels"])

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

def predict_encoder_one(model, tokenizer, premise, hypo, max_length=256):
    model.eval()
    enc = tokenizer(
        premise,
        hypo,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.argmax(logits, dim=-1).item()
    

Les performances sont évaluées sur l’ensemble de test afin de comparer les capacités de généralisation des deux encodeurs.

In [102]:
camembert_metrics = eval_encoder(model, tokenizer, hf_test)

print("Camembert Accuracy:", camembert_metrics["accuracy"])
print("Camembert Macro F1:", camembert_metrics["macro_f1"])

camemberta_metrics = eval_encoder(model_a, tokenizer_a, hf_test)

print("Camemberta Accuracy:", camemberta_metrics["accuracy"])
print("Camemberta Macro F1:", camemberta_metrics["macro_f1"])


100%|██████████| 78/78 [00:05<00:00, 13.29it/s]


Camembert Accuracy: 0.729718875502008
Camembert Macro F1: 0.7296866622760678


100%|██████████| 78/78 [00:07<00:00,  9.98it/s]

Camemberta Accuracy: 0.3321285140562249
Camemberta Macro F1: 0.18006169524993634


Les résultats sont stockés dans une structure commune afin de faciliter la comparaison avec les approches basées sur des modèles décodeurs et l’apprentissage en contexte.

In [103]:
results =[]
results.append({
    "model": "CamemBERT 2.0",
    "approach": "Encoder + LoRA",
    **camembert_metrics
})
results.append({
    "model": "CamemBERTa 2.0",
    "approach": "Encoder + LoRA",
    **camemberta_metrics
})

**MODULE 4 — Modèle décodeur LLaMA 3 + LoRA**

Le modèle LLaMA 3 est utilisé comme décodeur génératif. Le tokenizer associé au modèle Instruct est employé afin de respecter le format conversationnel attendu par le modèle.

In [104]:
#LLaMA 3 + LoRA
model_name_llama = "meta-llama/Llama-3.2-1B-Instruct"
# Chargement du tokenizer
tokenizer_llama = AutoTokenizer.from_pretrained(model_name_llama,token=hf_token)
tokenizer_llama.pad_token = tokenizer_llama.eos_token

# Chargement du modèle LLaMA
model_llama = AutoModelForCausalLM.from_pretrained(
    model_name_llama,
    device_map="auto",
    token=hf_token
).to(torch.float16)


La tâche de classification est reformulée sous forme d’instruction afin de l’adapter à un modèle décodeur génératif.

L’apprentissage supervisé consiste à entraîner le modèle à générer explicitement l’étiquette de la relation à partir du prompt.
Les labels sont donc intégrés directement dans la séquence cible.

In [105]:
def build_prompt(premise, hypo):
    return f"""Tu es un expert en compréhension du langage naturel.

Détermine la relation logique entre les deux phrases suivantes.

Phrase 1 : {premise}
Phrase 2 : {hypo}

La relation est l'une des suivantes :
- contradiction
- entailment
- neutral

Réponds uniquement par le nom de la relation.
"""


def preprocess_llama(batch):
    prompts = []
    for p, h, label in zip(batch["-e premise"], batch["hypo"], batch["labels"]):
        prompt = build_prompt(p, h)
        label_text = id2label[label]
        full_text = prompt + label_text
        prompts.append(full_text)

    tokenized = tokenizer_llama(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=256
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

Les données sont converties au format PyTorch afin d’être utilisées par le Trainer.

In [106]:
hf_train_llama = hf_train.map(preprocess_llama, batched=True)
hf_valid_llama = hf_valid.map(preprocess_llama, batched=True)

hf_train_llama.set_format("torch")
hf_valid_llama.set_format("torch")


Map:   0%|          | 0/4258 [00:00<?, ? examples/s]

Map:   0%|          | 0/752 [00:00<?, ? examples/s]

Les adaptateurs LoRA sont injectés dans les projections d’attention du décodeur afin de limiter le nombre de paramètres entraînables.
Un dropout plus faible est utilisé afin de préserver la stabilité de la génération auto-régressive.

In [107]:
lora_config_llama = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_llama = get_peft_model(model_llama, lora_config_llama)
model_llama.print_trainable_parameters()


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


Un taux d’apprentissage plus faible est utilisé pour le décodeur, celui-ci étant plus sensible aux variations de gradient.
L’accumulation de gradients permet de simuler un batch effectif plus important malgré les contraintes mémoire.

In [108]:
training_args_llama = TrainingArguments(
    output_dir="./llama3_lora",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, 
    num_train_epochs=1,
    logging_steps=50,
    load_best_model_at_end=None,
    metric_for_best_model=None,
    fp16=True,
    report_to="none"
)

trainer_llama = Trainer(
    model=model_llama,
    args=training_args_llama,
    train_dataset=hf_train_llama,
    eval_dataset=hf_valid_llama,
    processing_class=tokenizer_llama
)

trainer_llama.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
50,1.893200
100,0.686600
150,0.606400
200,0.606300
250,0.597600


TrainOutput(global_step=267, training_loss=0.8602632565444774, metrics={'train_runtime': 891.9518, 'train_samples_per_second': 4.774, 'train_steps_per_second': 0.299, 'total_flos': 6375800070733824.0, 'train_loss': 0.8602632565444774, 'epoch': 1.0})

La génération est volontairement limitée à quelques tokens et réalisée de manière déterministe afin de garantir une évaluation fiable.

In [109]:
def predict_relation(premise, hypo):
    prompt = build_prompt(premise, hypo)
    inputs = tokenizer_llama(prompt,add_special_tokens=False,return_tensors="pt").to(model_llama.device)
    outputs = model_llama.generate(**inputs,max_new_tokens=3,pad_token_id=tokenizer_llama.eos_token_id)
    decoded = tokenizer_llama.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return decoded.strip().lower().replace(".", "")


L’évaluation du modèle décodeur est effectuée a posteriori par génération des prédictions, puis comparaison avec les étiquettes de référence.

In [110]:
def normalize_label(text):
    text = text.lower()
    if "contradiction" in text:
        return "contradiction"
    if "entailment" in text:
        return "entailment"
    if "neutral" in text:
        return "neutral"
    return "neutral"

def eval_decoder(predict_fn, dataset):
    y_true, y_pred = [], []

    for example in tqdm(dataset):
        pred = predict_fn(example["-e premise"], example["hypo"])
        y_pred.append(pred)
        y_true.append(id2label[example["labels"]])

    y_pred_norm = [normalize_label(l) for l in y_pred]

    return {
        "accuracy": accuracy_score(y_true, y_pred_norm),
        "macro_f1": f1_score(y_true, y_pred_norm, average="macro")
    }

Les performances du modèle décodeur sont ajoutées au tableau comparatif final afin d’être confrontées aux approches basées sur des encodeurs.

In [ ]:
llama_metrics = eval_decoder(predict_relation, hf_test)
print("Llama Accuracy:", llama_metrics["accuracy"])
print("Llama Macro F1:", llama_metrics["macro_f1"])
results.append({
    "model": "LLaMA 3 1B",
    "approach": "Decoder + LoRA",
    **llama_metrics
})


100%|██████████| 2490/2490 [03:55<00:00, 10.56it/s]

Llama Accuracy: 0.3389558232931727
Llama Macro F1: 0.3004501783188834


**MODULE 5 — In-Context Learning (0-shot)**

Le prompt 0-shot consiste à décrire explicitement la tâche sans fournir d’exemples.
Le modèle doit inférer la relation uniquement à partir de ses connaissances acquises lors du pré-entraînement.

La génération est volontairement limitée à quelques tokens afin d’éviter toute dérive textuelle.
La sortie est tronquée pour ne conserver que la prédiction générée par le modèle, indépendamment du prompt d’entrée.

In [ ]:
# Prompt 0-shot

def build_prompt_0shot(premise, hypo):
    messages = [
        {"role": "system","content": "Tu es un expert en inférence textuelle."},
        {"role": "user","content": f"""
Phrase 1 : {premise}
Phrase 2 : {hypo}

Quelle est la relation logique entre ces deux phrases ?
Réponds uniquement par l'une des étiquettes suivantes :
entailment, contradiction ou neutral.
"""
        }
    ]
    return tokenizer_llama.apply_chat_template(
        messages,
        tokenize=False
    )

def predict_0shot(premise, hypo):
    prompt = build_prompt_0shot(premise, hypo)

    inputs = tokenizer_llama(prompt,return_tensors="pt",add_special_tokens=False).to(model_llama.device)

    outputs = model_llama.generate(**inputs,max_new_tokens=3,pad_token_id=tokenizer_llama.eos_token_id)

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer_llama.decode(generated, skip_special_tokens=True)

    return decoded.strip().lower().replace(".", "")

Les performances du modèle en mode 0-shot sont évaluées sur l’ensemble de test à l’aide des mêmes métriques que pour les approches entraînées.

Les résultats sont ajoutés au tableau comparatif final afin de mesurer l’impact de l’apprentissage en contexte par rapport à l’ajustement supervisé.

In [ ]:
metrics_0shot = eval_decoder(predict_0shot,hf_test)

print("0-shot Accuracy:", metrics_0shot["accuracy"])
print("0-shot Macro F1:", metrics_0shot["macro_f1"])
results.append({
    "model": "LLaMA 3 1B",
    "approach": "In-context (0-shot)",
    **metrics_0shot
})


100%|██████████| 2490/2490 [04:42<00:00,  8.82it/s]

0-shot Accuracy: 0.3333333333333333
0-shot Macro F1: 0.16666666666666666


**MODULE 6 — In-Context Learning (few-shot)**

En mode few-shot, quelques exemples représentatifs sont fournis au modèle afin de l’aider à inférer la tâche par analogie.
Ces exemples couvrent les trois relations possibles, permettant au modèle de mieux calibrer sa prédiction.

Comme pour le mode 0-shot, la génération est volontairement limitée à quelques tokens et réalisée de manière déterministe afin de garantir une évaluation stable et reproductible.

In [ ]:
#Prompt few-shot

def build_prompt_fewshot(premise, hypo):
    messages = [
    {
        "role": "system",
        "content": "Tu es un expert en inférence textuelle (Natural Language Inference)."
    },
    {
        "role": "user",
        "content": f"""
Voici quelques exemples d’inférence textuelle.

Exemple 1 :
Phrase 1 : Duy a acheté un billet pour le concert.
Phrase 2 : Duy va assister au concert.
Relation : entailment

Exemple 2 :
Phrase 1 : Le train est arrivé à l’heure.
Phrase 2 : Le train est en retard.
Relation : contradiction

Exemple 3 :
Phrase 1 : Han travaille dans une banque.
Phrase 2 : Han aime voyager.
Relation : neutral

Maintenant, détermine la relation logique entre les deux phrases suivantes.

Phrase 1 : {premise}
Phrase 2 : {hypo}

Réponds uniquement par un seul mot parmi :
entailment, contradiction ou neutral.
"""
    }
]
    return tokenizer_llama.apply_chat_template(
        messages,
        tokenize=False
    )
    
def predict_fewshot(premise, hypo):
    prompt = build_prompt_fewshot(premise, hypo)

    inputs = tokenizer_llama(prompt,return_tensors="pt",add_special_tokens=False).to(model_llama.device)

    outputs = model_llama.generate(**inputs,max_new_tokens=3,pad_token_id=tokenizer_llama.eos_token_id)

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer_llama.decode(generated, skip_special_tokens=True)

    return decoded.strip().lower().replace(".", "")

L’ajout de quelques exemples améliore généralement les performances par rapport au mode 0-shot, en permettant au modèle de mieux comprendre la structure de la tâche.

Les résultats obtenus en mode few-shot sont intégrés au tableau comparatif final afin d’évaluer l’apport de l’apprentissage en contexte.

In [ ]:
fewshot_metrics = eval_decoder(predict_fewshot,hf_test)

print("Few-shot Accuracy:", fewshot_metrics["accuracy"])
print("Few-shot Macro F1:", fewshot_metrics["macro_f1"])
results.append({
    "model": "LLaMA 3 1B",
    "approach": "In-context (few-shot)",
    **fewshot_metrics
})


100%|██████████| 2490/2490 [06:38<00:00,  6.24it/s]

Few-shot Accuracy: 0.3333333333333333
Few-shot Macro F1: 0.16666666666666666


**MODULE 7 — In-Context Learning (Chain-of-Thought)**

Le prompt Chain-of-Thought encourage le modèle à expliciter son raisonnement avant de fournir la prédiction finale.
Cette approche vise à améliorer la cohérence des décisions dans les cas ambigus, au prix d’un coût de génération plus élevé.

La prédiction est obtenue en deux étapes : (1) génération d’un raisonnement court, (2) extraction de l’étiquette finale depuis le texte généré.
Une stratégie de repli vers la classe neutral est appliquée lorsque la génération ne contient pas explicitement d’étiquette valide.

In [ ]:
#CHAIN-OF-THOUGHT (CoT)

def build_prompt_cot(premise, hypo):
    messages = [
        {"role": "system","content": "Tu es un expert en raisonnement logique."},
        {"role": "user",
         "content": f"""
Analyse les deux phrases étape par étape, puis donne la relation finale.

Phrase 1 : {premise}
Phrase 2 : {hypo}

Explique brièvement ton raisonnement,
puis termine par une ligne :
Relation : <entailment | contradiction | neutral>
"""
        }
    ]

    return tokenizer_llama.apply_chat_template(
        messages,
        tokenize=False
    )

def predict_cot(premise, hypo):
    prompt = build_prompt_cot(premise, hypo)
    inputs = tokenizer_llama(prompt,return_tensors="pt",add_special_tokens=False).to(model_llama.device)

    outputs = model_llama.generate(**inputs,max_new_tokens=30,pad_token_id=tokenizer_llama.eos_token_id)

    decoded = tokenizer_llama.decode(outputs[0][inputs["input_ids"].shape[-1]:],skip_special_tokens=True).lower().strip()

    return decoded.strip().lower().replace(".", "")

Les performances en mode CoT sont évaluées avec le même protocole que les autres approches, garantissant une comparaison équitable.

In [ ]:
cot_metrics = eval_decoder(predict_cot,hf_test)

print("CoT Accuracy:", cot_metrics["accuracy"])
print("CoT Macro F1:", cot_metrics["macro_f1"])
results.append({
    "model": "LLaMA 3 1B",
    "approach": "In-context (Chain-of-Thought)",
    **cot_metrics
})


100%|██████████| 2490/2490 [34:44<00:00,  1.19it/s]

CoT Accuracy: 0.3345381526104418
CoT Macro F1: 0.236884688397655


Les résultats sont synthétisés dans un tableau unique afin de comparer directement les approches encodeur (ajustement LoRA), décodeur (ajustement LoRA) et apprentissage en contexte (0-shot, few-shot, CoT).

In [ ]:
results_all = pd.DataFrame(results)
results_all

,model,approach,accuracy,macro_f1
0,CamemBERT 2.0,Encoder + LoRA,0.729719,0.729687
1,CamemBERTa 2.0,Encoder + LoRA,0.332129,0.180062
2,LLaMA 3 1B,Decoder + LoRA,0.338956,0.300450
3,LLaMA 3 1B,In-context (0-shot),0.333333,0.166667
4,LLaMA 3 1B,In-context (few-shot),0.333333,0.166667
5,LLaMA 3 1B,In-context (Chain-of-Thought),0.334538,0.236885


In [ ]:
synthetic_examples = [
    {
        "premise": "Marie a acheté une voiture rouge hier.",
        "hypo": "Marie possède une voiture.",
        "label": "entailment"
    },
    {
        "premise": "La réunion a été annulée à cause de la pluie.",
        "hypo": "La réunion a eu lieu comme prévu.",
        "label": "contradiction"
    },
    {
        "premise": "Paul lit un livre dans le salon.",
        "hypo": "Paul aime les romans de science-fiction.",
        "label": "neutral"
    },
    {
        "premise": "Paul a acheté une voiture rouge hier matin.",
        "hypo": "Paul a acheté une voiture.",
        "label": "entailment"
    },
    {
        "premise": "Le magasin est fermé le dimanche.",
        "hypo": "Le magasin est ouvert tous les jours de la semaine.",
        "label": "contradiction"
    },
    {
        "premise": "Marie lit un roman dans le train.",
        "hypo": "Marie se rend à Paris.",
        "label": "neutral"
    },
    {
        "premise": "Tous les étudiants de la classe ont réussi l’examen.",
        "hypo": "Certains étudiants de la classe ont réussi l’examen.",
        "label": "entailment"
    },
    {
        "premise": "Le concert a commencé à 20 heures.",
        "hypo": "Le concert a duré deux heures.",
        "label": "neutral"
    },
    {
        "premise": "Jean n’a jamais voyagé en avion.",
        "hypo": "Jean a pris l’avion au moins une fois.",
        "label": "contradiction"
    }]


In [ ]:
rows = []

for ex in synthetic_examples:
    premise = ex["premise"]
    hypo = ex["hypo"]
    label = ex["label"]

    rows.append({
        "Label (true)": label,

        "CamemBERT 2.0": id2label[predict_encoder_one(model, tokenizer, premise, hypo)],
        "CamemBERTa 2.0": id2label[predict_encoder_one(model_a, tokenizer_a, premise, hypo)],

        "LLaMA 3 1B (Dec+LoRA)": normalize_label(predict_relation(premise, hypo)),
        "LLaMA 3 1B (0-shot)": normalize_label(predict_0shot(premise, hypo)),
        "LLaMA 3 1B (few-shot)": normalize_label(predict_fewshot(premise, hypo)),
        "LLaMA 3 1B (CoT)": normalize_label(predict_cot(premise, hypo)),
    })

df_qualitative = pd.DataFrame(rows)
df_qualitative


,Label (true),CamemBERT 2.0,CamemBERTa 2.0,LLaMA 3 1B (Dec+LoRA),LLaMA 3 1B (0-shot),LLaMA 3 1B (few-shot),LLaMA 3 1B (CoT)
0,entailment,entailment,contradiction,entailment,neutral,neutral,neutral
1,contradiction,contradiction,contradiction,entailment,neutral,neutral,neutral
2,neutral,neutral,contradiction,entailment,neutral,neutral,neutral
3,entailment,entailment,contradiction,entailment,neutral,neutral,neutral
4,contradiction,contradiction,contradiction,neutral,neutral,neutral,neutral
5,neutral,neutral,contradiction,neutral,neutral,neutral,neutral
6,entailment,entailment,contradiction,entailment,neutral,neutral,neutral
7,neutral,entailment,contradiction,entailment,neutral,neutral,neutral
8,contradiction,contradiction,contradiction,entailment,neutral,neutral,neutral
